# 示例: 1. SCS产流模型示例

**脚本:** `examples/run_scs_example.py`

## 目的

演示 `SCSCurveNumberModule` 的用法。脚本定义了一个CN值，然后输入一系列降雨脉冲，输出由SCS公式计算得到的有效径流深。

## 1. 环境设置

首先，我们导入所需的库，并把项目的根目录添加到Python的搜索路径中，以便能顺利找到我们自己开发的`hydro_model`模块。

In [ ]:
import numpy as np
import sys
import os
import matplotlib.pyplot as plt

# 将项目根目录添加到Python路径
sys.path.insert(0, os.path.abspath(os.path.join(os.path.dirname('__file__'), '..')))
from hydro_model.model import HydrologicalModel
from hydro_model.runoff import SCSCurveNumberModule
from hydro_model.routing import SimpleRouting

## 2. 定义参数

对于SCS产流模型，核心参数是`CN`（Curve Number）。我们这里使用一个典型的`CN=85`。其他参数`k_q`和`k_s`是用于汇流模块的，这里我们使用一个`SimpleRouting`模块，所以也需要提供这些参数。

In [ ]:
params = {'CN': 85, 'k_q': 0.8, 'k_s': 0.1}

## 3. 创建模型组件

我们分别创建产流模块（`SCSCurveNumberModule`）和汇流模块（`SimpleRouting`）。`SimpleRouting`只是一个简单的线性水库模型，用于将产流转化为流量。

In [ ]:
scs_runoff_module = SCSCurveNumberModule(**params)
simple_routing_module = SimpleRouting(**params)

# 4. 组合成完整的水文模型
model = HydrologicalModel(
    runoff_module=scs_runoff_module,
    routing_module=simple_routing_module
)

## 5. 运行模拟

我们使用一个简单的降雨序列作为输入，然后循环调用模型的`run`方法，计算每个时间步的径流深。

In [ ]:
sample_rainfall = [0, 5, 10, 30, 50, 20, 10, 5, 0]
runoff_results = []
total_runoff = 0

for t, rain in enumerate(sample_rainfall):
    runoff_mm = model.run(rainfall=rain, pet=0)
    runoff_results.append(runoff_mm)
    total_runoff += runoff_mm

print("--- 模拟结果 ---")
print("时间 | 降雨 | 径流 (mm)")
print("-------------------------")
for i in range(len(sample_rainfall)):
    print(f"{i:4d} | {sample_rainfall[i]:5.2f} | {runoff_results[i]:10.4f}")
print("-------------------------")
print(f"总径流: {total_runoff:.4f} mm")

## 6. 可视化结果

为了更直观地展示结果，我们绘制了降雨和产流过程线。

In [ ]:
time_steps = np.arange(len(sample_rainfall))

fig, ax1 = plt.subplots(figsize=(10, 6))

color = 'tab:blue'
ax1.set_xlabel('Time Step')
ax1.set_ylabel('Runoff (mm)', color=color)
ax1.plot(time_steps, runoff_results, color=color, marker='o', label='Runoff')
ax1.tick_params(axis='y', labelcolor=color)

ax2 = ax1.twinx()  # instantiate a second axes that shares the same x-axis

color = 'tab:grey'
ax2.set_ylabel('Rainfall (mm)', color=color)  # we already handled the x-label with ax1
ax2.bar(time_steps, sample_rainfall, color=color, alpha=0.6, label='Rainfall')
ax2.tick_params(axis='y', labelcolor=color)
ax2.invert_yaxis()

fig.tight_layout()  # otherwise the right y-label is slightly clipped
plt.title('SCS Model: Rainfall and Runoff')
plt.show()